# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

For my **Refresh / Content Opportunity Scoring** lane, the starter data contract is:

**One row = one pseudonymized content page/content item.**

The file I am using is `data/raw/content_refresh_anonymized.csv`. It is a starter teaching slice with page-level metrics, not raw URLs, titles, client names, or private queries. Most activity columns use a **trailing 90-day window**, such as `impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`, `engagement_rate`, and `scroll_rate`. The trend inputs use two shorter comparison windows: the most recent 30 days and the previous 30 days.

My Week 3 output is not a model yet. It is a checked promise about what the rows and columns mean before I build any baseline or ML score.


In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
data_path = next(path for path in candidates if path.exists())
df = pd.read_csv(data_path)

df['is_declining_proxy'] = df['trend_direction'].eq('down').astype(int)

unit_check = pd.DataFrame({
    'contract_claim': [
        'one row is one pseudonymized content item/page',
        'starter slice rows',
        'unique content_id values',
        'unique client_id values',
        'rows with duplicate content_id',
        'rows with impressions_90d > 0',
        'rows with content_age_days >= 90',
    ],
    'verified_value': [
        'content_id is unique per row',
        len(df),
        df['content_id'].nunique(),
        df['client_id'].nunique(),
        int(df.duplicated('content_id').sum()),
        int((df['impressions_90d'] > 0).sum()),
        int((df['content_age_days'] >= 90).sum()),
    ],
})

unit_check


,contract_claim,verified_value
0,one row is one pseudonymized content item/page,content_id is unique per row
1,starter slice rows,30000
2,unique content_id values,30000
3,unique client_id values,32
4,rows with duplicate content_id,0
5,rows with impressions_90d > 0,30000
6,rows with content_age_days >= 90,30000


## 2. Fields: feature / label / context / excluded

I sort fields into four buckets before modeling:

- **Feature:** safe model input, knowable before the scoring decision.
- **Label / proxy:** the thing I am trying to predict or fields used to create that proxy. These must not become normal features.
- **Context:** useful for grouping, splitting, checking, or tracing rows, but not for the model to learn from directly.
- **Excluded:** not planned for this lane, too risky, or not useful for the decision.

For this starter notebook, the proxy is `is_declining_proxy = trend_direction == "down"`. Because `trend_direction` is computed from trend movement, I treat `trend_direction`, `trend_pct`, and the 30-day comparison fields as proxy/source fields instead of model features.


In [2]:
field_contract = pd.DataFrame(
    [
        ('content_id', 'context', 'row id / grouping only', 'pseudonymized id; never a model feature'),
        ('client_id', 'context', 'grouping / validation split', 'pseudonymized id; use for client-holdout checks, not as a feature'),
        ('search_volume', 'feature', 'keyword-context estimate', 'safe observed/context signal when present'),
        ('competition', 'feature', 'keyword-context estimate', 'safe observed/context signal when present'),
        ('competition_level', 'feature', 'keyword-context category', 'safe observed/context signal when present'),
        ('cpc', 'feature', 'keyword-context estimate', 'safe observed/context signal when present'),
        ('content_type', 'feature', 'content metadata', 'safe category but check missingness patterns'),
        ('main_intent', 'feature', 'content metadata', 'safe category when present'),
        ('word_count', 'feature', 'content property', 'safe content-depth signal when measured'),
        ('char_count', 'feature', 'content property', 'safe content-depth signal when measured'),
        ('provider_used', 'excluded', 'generation metadata', 'not part of the reviewer decision; could add bias/noise'),
        ('model_used', 'excluded', 'generation metadata', 'not part of the reviewer decision; could add bias/noise'),
        ('impressions_90d', 'feature', 'trailing 90-day activity', 'visibility/demand signal'),
        ('clicks_90d', 'feature', 'trailing 90-day activity', 'search click signal'),
        ('pageviews_90d', 'feature', 'trailing 90-day activity', 'site traffic signal'),
        ('sessions_90d', 'feature', 'trailing 90-day activity', 'site traffic signal'),
        ('users_90d', 'feature', 'trailing 90-day activity', 'site traffic signal'),
        ('engaged_sessions_90d', 'feature', 'trailing 90-day activity', 'engagement signal'),
        ('ai_sessions_90d', 'feature', 'trailing 90-day activity', 'AI referral click-through signal, sparse and not AI citations'),
        ('scroll_events_90d', 'feature', 'trailing 90-day activity', 'engagement/scroll signal'),
        ('days_with_impressions', 'feature', 'trailing 90-day activity', 'visibility consistency signal'),
        ('days_with_sessions', 'feature', 'trailing 90-day activity', 'traffic consistency signal'),
        ('impressions_last_30d', 'label / proxy source', 'latest 30-day trend input', 'used to create trend proxy; not a normal feature for this proxy'),
        ('clicks_last_30d', 'label / proxy source', 'latest 30-day trend input', 'trend/source context; not a normal feature for this proxy'),
        ('sessions_last_30d', 'label / proxy source', 'latest 30-day trend input', 'trend/source context; not a normal feature for this proxy'),
        ('impressions_prev_30d', 'label / proxy source', 'previous 30-day trend input', 'used to create trend proxy; not a normal feature for this proxy'),
        ('clicks_prev_30d', 'label / proxy source', 'previous 30-day trend input', 'trend/source context; not a normal feature for this proxy'),
        ('sessions_prev_30d', 'label / proxy source', 'previous 30-day trend input', 'trend/source context; not a normal feature for this proxy'),
        ('content_age_days', 'feature', 'content age', 'safe freshness/lifecycle signal'),
        ('age_tier', 'feature', 'derived from content age', 'safe interpretable bucket'),
        ('age_tier_order', 'feature', 'derived from content age', 'safe ordered age bucket'),
        ('days_since_last_update', 'feature', 'freshness', 'safe staleness signal'),
        ('freshness_tier', 'feature', 'derived from freshness', 'safe interpretable bucket'),
        ('word_count_tier', 'feature', 'derived from word_count', 'safe content-depth bucket when measured'),
        ('char_count_tier', 'feature', 'derived from char_count', 'safe content-depth bucket when measured'),
        ('ctr', 'feature', 'trailing 90-day derived rate', 'safe rate; value is percentage points, e.g. 0.76 means 0.76%'),
        ('avg_position', 'feature', 'trailing 90-day search metric', 'safe ranking-position signal; 0 means no data'),
        ('engagement_rate', 'feature', 'trailing 90-day derived rate', 'safe engagement signal; percentage points'),
        ('scroll_rate', 'feature', 'trailing 90-day derived rate', 'safe but can exceed 100 because multiple scroll events can occur'),
        ('ai_traffic_pct', 'feature', 'trailing 90-day derived rate', 'safe but sparse; not AI visibility/citation proof'),
        ('impression_tier', 'feature', 'derived from impressions', 'safe interpretable visibility bucket'),
        ('position_tier', 'feature', 'derived from avg_position', 'safe interpretable position bucket'),
        ('trend_direction', 'label / proxy', 'current trend bucket', 'source of is_declining_proxy; never a feature for this proxy'),
        ('trend_pct', 'label / proxy source', 'current trend calculation', 'feeds trend bucket; never a feature for this proxy'),
        ('is_declining_proxy', 'label / proxy', 'starter target proxy', '1 when trend_direction == down; useful for practice, not causal proof'),
    ],
    columns=['field', 'role', 'time_or_use', 'contract_reason'],
)

field_contract


,field,role,time_or_use,contract_reason
0,content_id,context,row id / grouping only,pseudonymized id; never a model feature
1,client_id,context,grouping / validation split,pseudonymized id; use for client-holdout check...
2,search_volume,feature,keyword-context estimate,safe observed/context signal when present
3,competition,feature,keyword-context estimate,safe observed/context signal when present
4,competition_level,feature,keyword-context category,safe observed/context signal when present
5,cpc,feature,keyword-context estimate,safe observed/context signal when present
6,content_type,feature,content metadata,safe category but check missingness patterns
7,main_intent,feature,content metadata,safe category when present
8,word_count,feature,content property,safe content-depth signal when measured
9,char_count,feature,content property,safe content-depth signal when measured


## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below turn the contract into evidence. I verify the row grain, counts, proxy distribution, missing values, and a few important field rules from the data dictionary.

Key checks I care about:

- `content_id` should be unique if one row is one content item.
- The starter slice should have 30,000 rows and 32 pseudonymized clients.
- The proxy should have both positive and negative rows.
- Missingness should be inspected before filling blanks.
- Rate fields are percentage points; `avg_position = 0` means no position data.


In [3]:
grain_summary = pd.DataFrame({
    'check': [
        'rows',
        'columns before proxy',
        'unique content_id',
        'duplicate content_id rows',
        'unique clients',
        'declining proxy rows',
        'declining proxy share',
        'avg_position == 0 rows',
        'scroll_rate > 100 rows',
        'ai_traffic_pct > 100 rows',
    ],
    'value': [
        len(df),
        len([c for c in df.columns if c != 'is_declining_proxy']),
        df['content_id'].nunique(),
        int(df.duplicated('content_id').sum()),
        df['client_id'].nunique(),
        int(df['is_declining_proxy'].sum()),
        round(df['is_declining_proxy'].mean(), 3),
        int((df['avg_position'] == 0).sum()),
        int((df['scroll_rate'] > 100).sum()),
        int((df['ai_traffic_pct'] > 100).sum()),
    ],
})

missing_fields = [
    'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent',
    'word_count', 'char_count', 'word_count_tier', 'char_count_tier',
    'scroll_rate', 'ai_traffic_pct', 'trend_pct',
]
missing_summary = (
    df[missing_fields]
    .isna()
    .sum()
    .rename('missing_rows')
    .reset_index()
    .rename(columns={'index': 'field'})
)
missing_summary['missing_share'] = (missing_summary['missing_rows'] / len(df)).round(3)

missing_by_content_type = (
    df.groupby('content_type')[['search_volume', 'word_count', 'scroll_rate', 'trend_pct']]
    .apply(lambda part: part.isna().mean())
    .round(3)
    .reset_index()
)

window_summary = pd.DataFrame({
    'field_group': [
        '90-day activity fields',
        'latest 30-day trend fields',
        'previous 30-day trend fields',
        'content age / freshness fields',
    ],
    'fields': [
        'impressions_90d, clicks_90d, sessions_90d, ctr, avg_position, engagement_rate, scroll_rate',
        'impressions_last_30d, clicks_last_30d, sessions_last_30d',
        'impressions_prev_30d, clicks_prev_30d, sessions_prev_30d',
        'content_age_days, days_since_last_update',
    ],
    'contract_window': [
        'trailing 90 days ending at export time',
        'most recent 30 days inside the starter window',
        'days 31-60 before export inside the starter window',
        'age/freshness as of export time',
    ],
})

print('GRAIN AND COUNT CHECKS')
print(grain_summary.to_string(index=False))
print('\nMISSING VALUE CHECKS')
print(missing_summary.to_string(index=False))
print('\nMISSINGNESS BY CONTENT TYPE')
print(missing_by_content_type.to_string(index=False))
print('\nWINDOW CONTRACT')
print(window_summary.to_string(index=False))


GRAIN AND COUNT CHECKS
                    check     value
                     rows 30000.000
     columns before proxy    44.000
        unique content_id 30000.000
duplicate content_id rows     0.000
           unique clients    32.000
     declining proxy rows 16262.000
    declining proxy share     0.542
   avg_position == 0 rows  1205.000
   scroll_rate > 100 rows   119.000
ai_traffic_pct > 100 rows    23.000

MISSING VALUE CHECKS
            field  missing_rows  missing_share
    search_volume          2468          0.082
      competition          2468          0.082
competition_level          2610          0.087
              cpc          2468          0.082
      main_intent          2374          0.079
       word_count          7699          0.257
       char_count          7699          0.257
  word_count_tier          7699          0.257
  char_count_tier          7699          0.257
      scroll_rate           125          0.004
   ai_traffic_pct             0          0

## 4. Data limits

This data can support a careful review-prioritization project, but it cannot answer everything.

- It cannot prove that editing or refreshing a page **caused** recovery, because this is observational data, not an experiment.
- It cannot reveal real client names, domains, URLs, article titles, or private queries; those are intentionally removed or pseudonymized.
- The starter proxy `is_declining_proxy` is useful for practice, but it is not a future-looking outcome. A stronger capstone label should use prior-window features and a later target window.
- `trend_direction`, `trend_pct`, and the 30-day comparison fields are dangerous as features when predicting the decline proxy because they help define the answer.
- Missingness is not random. Some keyword/content fields are missing by content type, so blind `fillna(0)` could accidentally teach the model a category pattern.
- `avg_position = 0` means no position data, not search position zero.
- Rate columns are percentage points. For example, `ctr = 0.76` means 0.76%, not 76%.
- AI fields show click-through sessions from AI referrers. They do not prove AI citations, AI rankings, or AI visibility.

Because of these limits, the output should be a **decision-support review queue**, not an automatic claim that a page must be changed.


In [4]:
limits_check = pd.DataFrame({
    'limit_or_caution': [
        'proxy is not future-looking',
        'trend fields are proxy leakage risks',
        'keyword missingness is patterned',
        'word_count missingness is patterned',
        'avg_position zero is no-data',
        'AI sessions are sparse',
    ],
    'evidence_from_starter_data': [
        'is_declining_proxy is defined from trend_direction in this notebook',
        'trend_direction/trend_pct and 30-day comparison fields are labeled as proxy/source fields',
        f"search_volume missing rows: {int(df['search_volume'].isna().sum())}",
        f"word_count missing rows: {int(df['word_count'].isna().sum())}",
        f"avg_position == 0 rows: {int((df['avg_position'] == 0).sum())}",
        f"rows with ai_sessions_90d > 0: {int((df['ai_sessions_90d'] > 0).sum())}",
    ],
    'decision_for_my_contract': [
        'use as starter proxy only; prefer future-window target later',
        'do not use these as normal features for this proxy',
        'inspect missingness before imputation',
        'inspect missingness before imputation',
        'treat zero as missing/no-position context',
        'use cautiously; do not claim AI rankings/citations',
    ],
})

limits_check


,limit_or_caution,evidence_from_starter_data,decision_for_my_contract
0,proxy is not future-looking,is_declining_proxy is defined from trend_direc...,use as starter proxy only; prefer future-windo...
1,trend fields are proxy leakage risks,trend_direction/trend_pct and 30-day compariso...,do not use these as normal features for this p...
2,keyword missingness is patterned,search_volume missing rows: 2468,inspect missingness before imputation
3,word_count missingness is patterned,word_count missing rows: 7699,inspect missingness before imputation
4,avg_position zero is no-data,avg_position == 0 rows: 1205,treat zero as missing/no-position context
5,AI sessions are sparse,rows with ai_sessions_90d > 0: 1930,use cautiously; do not claim AI rankings/citat...


## Self-check

Before submitting, I checked each line honestly:

- [x] Every section above is filled with markdown thinking and code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are included.
- [x] My claims use careful words: observed, measured, proxy, directional, decision-support.
- [x] The field contract separates features, label/proxy fields, context fields, and excluded fields.
- [x] The work lives under `work/notebooks/`; after committing and pushing, I can submit my repo URL on the card.
